In [1]:
import torch
from torch import nn
from d2l import torch as d2l
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim

In [20]:
## THIS CODE IS TO CREATE THE TENSOR FOR THE TRAINING DATA ##
jw_df = pd.read_csv('train_set.csv')

# Pivot to create a time series for each (x, y)
jw_pivot = jw_df.pivot(index=['x', 'y'], columns='Date', values='Value')
print (jw_pivot)

# Reset index to keep (x, y) as columns
jw_pivot = jw_pivot.reset_index()

# Convert time columns back into a NumPy array
ts_jw = jw_pivot.iloc[:, 2:].values  # Ignore first two columns (x, y)
loc_jw = jw_pivot.iloc[:, :2].values  # Store coordinates

# Standardization with Z-score normalisation
jw_mean_LST = ts_jw.mean()
jw_std_LST = ts_jw.std()
ts_jw = (ts_jw - jw_mean_LST) / jw_std_LST 

# Convert to PyTorch tensor wiith extra dimension for LST
X_train_tensor_jw = torch.tensor(ts_jw, dtype=torch.float32).unsqueeze(-1)
print(X_train_tensor_jw.shape)

X_train_jw = X_train_tensor_jw
y_train_jw = X_train_tensor_jw[:, 1:, :]

Date                 Jan-Feb 2001  Jan-Feb 2002  Jan-Feb 2003  Jan-Feb 2004  \
x          y                                                                  
103.964566 1.350459     29.400195     29.891817     29.542257     24.297864   
           1.351269     29.554990     30.108200     29.359647     24.140527   
           1.352079     28.643756     29.394887     29.428029     24.358701   
           1.352889     28.540931     29.668894     29.712394     24.468010   
103.965377 1.348838     29.083580     29.845631     30.009634     25.069162   
...                           ...           ...           ...           ...   
104.032621 1.360991     25.516672     27.596076     27.428149     23.452554   
104.033431 1.358560     24.588188     26.992424     25.829845     22.441462   
           1.359371     26.109646     28.303498     28.434340     23.432783   
           1.360181     25.387832     28.316208     28.441986     23.521608   
           1.360991     25.367211     26.786237     

In [21]:
print(torch.isnan(X_train_jw).sum())  # Check for NaNs in X_train_jw
print(torch.isnan(y_train_jw).sum())  # Check for NaNs in y_train_jw

tensor(0)
tensor(0)


#### Without weight decay and dropout

In [22]:
## THIS CODE IS TO DEFINE THE LSTM MODEL ##

class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=3, output_size=1):
        super(LSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)  # Fully connected layer

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # LSTM output
        out = self.fc(lstm_out)  # Fully connected layer for final prediction
        return out

In [23]:
## THIS CODE IS TO TRAIN THE MODEL ##
torch.manual_seed(5188)

model = LSTMPredictor() # Model
criterion = nn.MSELoss() # Loss function to track performance over epochs
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer (can be switched)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    predictions = model(X_train_jw)

    loss = criterion(predictions[:, :-1, :], y_train_jw)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0: # Print loss every 10 epochs
        print(f"Epoch {epoch}/{num_epochs}, Loss: {loss.item():.4f}")

Epoch 0/100, Loss: 1.0082
Epoch 10/100, Loss: 0.9866
Epoch 20/100, Loss: 0.9414
Epoch 30/100, Loss: 0.9111
Epoch 40/100, Loss: 0.9006
Epoch 50/100, Loss: 0.8959
Epoch 60/100, Loss: 0.8907
Epoch 70/100, Loss: 0.8838
Epoch 80/100, Loss: 0.8766
Epoch 90/100, Loss: 0.8602


In [32]:
## THIS CODE IS FOR FORECASTING##

model.eval()
torch.manual_seed(5188)

with torch.no_grad():
    X_pred_jw = model(X_train_jw)  # Forecast next steps
    X_pred_jw= X_pred_jw[:, -12:, :]  # Extract last 12 bimonthly periods (2023-2024)

# Convert predictions to a NumPy array
X_pred_jw_np = X_pred_jw.squeeze().cpu().numpy()

# **Denormalize predictions**
X_pred_jw_np = (X_pred_jw_np * jw_std_LST) + jw_mean_LST  # Convert back to original from scaled values

# Define bimonthly periods
bimonthly_periods = [
    "Jan-Feb 2023", "Mar-Apr 2023", "May-Jun 2023", "Jul-Aug 2023", "Sep-Oct 2023", "Nov-Dec 2023",
    "Jan-Feb 2024", "Mar-Apr 2024", "May-Jun 2024", "Jul-Aug 2024", "Sep-Oct 2024", "Nov-Dec 2024"
]

# Create a long-format DataFrame
jw_pred_df = pd.DataFrame({
    "x": np.repeat(loc_jw[:, 0], len(bimonthly_periods)),  # Use stored locations
    "y": np.repeat(loc_jw[:, 1], len(bimonthly_periods)),  
    "Date": bimonthly_periods * len(loc_jw),
    "Predicted_LST": X_pred_jw_np.flatten()  # Store denormalized values
})

jw_pred_df.to_csv('jw_pred_long.csv', index=False) # Save to CSV for RMSE

In [34]:
## THIS CODE IS TO FIND THE RMSE BETWEEN TRUE AND PREDICTED LST ##

true_df = pd.read_csv("test_set.csv")
pred_df = pd.read_csv("jw_pred_long.csv")

print(true_df.head())
print(pred_df.head())

merged_df = true_df.merge(pred_df, on=["x", "y", "Date"], suffixes=("_true", "_pred"))
print(merged_df.head())

# Compute RMSE
rmse = np.sqrt(np.mean((merged_df["Value"] - merged_df["Predicted_LST"]) ** 2))
print(f"RMSE between true and predicted LST for Jurong West is: {rmse:.4f}")

            x         y          Date      Value
0  103.964566  1.350459  Jan-Feb 2023  19.300148
1  103.964566  1.351269  Jan-Feb 2023  19.300060
2  103.964566  1.352079  Jan-Feb 2023  19.299971
3  103.964566  1.352889  Jan-Feb 2023  19.299883
4  103.965377  1.348838  Jan-Feb 2023  19.300281
            x         y          Date  Predicted_LST
0  103.964566  1.350459  Jan-Feb 2023      24.993717
1  103.964566  1.350459  Mar-Apr 2023      25.426064
2  103.964566  1.350459  May-Jun 2023      24.788220
3  103.964566  1.350459  Jul-Aug 2023      25.540777
4  103.964566  1.350459  Sep-Oct 2023      25.308475
            x         y          Date      Value  Predicted_LST
0  103.964566  1.350459  Jan-Feb 2023  19.300148      24.993717
1  103.964566  1.351269  Jan-Feb 2023  19.300060      25.041248
2  103.964566  1.352079  Jan-Feb 2023  19.299971      24.830894
3  103.964566  1.352889  Jan-Feb 2023  19.299883      24.970928
4  103.965377  1.348838  Jan-Feb 2023  19.300281      25.005123
RMSE

#### With weight decay and dropout

In [35]:
# Define LSTM model with Dropout
class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=3, output_size=1, dropout=0.2):
        super(LSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out)
        return out

In [36]:
# Initialize model
torch.manual_seed(5188)
model = LSTMPredictor()

# Define loss function and optimizer with weight decay
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)  # Added weight decay

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_jw)
    loss = criterion(predictions[:, :-1, :], y_train_jw)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{num_epochs}, Loss: {loss.item():.4f}")

Epoch 0/100, Loss: 1.0082
Epoch 10/100, Loss: 0.9868
Epoch 20/100, Loss: 0.9417
Epoch 30/100, Loss: 0.9120
Epoch 40/100, Loss: 0.9017
Epoch 50/100, Loss: 0.8976
Epoch 60/100, Loss: 0.8933
Epoch 70/100, Loss: 0.8870
Epoch 80/100, Loss: 0.8813
Epoch 90/100, Loss: 0.8720


In [37]:
# Forecasting
model.eval()
torch.manual_seed(5188)
with torch.no_grad():
    X_pred_jw = model(X_train_jw)
    X_pred_jw = X_pred_jw[:, -12:, :]

# Denormalize predictions
X_pred_jw_np = (X_pred_jw.squeeze().cpu().numpy() * jw_std_LST) + jw_mean_LST

# Define bimonthly periods
bimonthly_periods = [
    "Jan-Feb 2023", "Mar-Apr 2023", "May-Jun 2023", "Jul-Aug 2023", "Sep-Oct 2023", "Nov-Dec 2023",
    "Jan-Feb 2024", "Mar-Apr 2024", "May-Jun 2024", "Jul-Aug 2024", "Sep-Oct 2024", "Nov-Dec 2024"
]

# Create DataFrame for predictions
jw_pred_df = pd.DataFrame({
    "x": np.repeat(loc_jw[:, 0], len(bimonthly_periods)),
    "y": np.repeat(loc_jw[:, 1], len(bimonthly_periods)),  
    "Date": bimonthly_periods * len(loc_jw),
    "Predicted_LST": X_pred_jw_np.flatten()
})

# Save predictions to CSV
jw_pred_df.to_csv('jw_pred_long_do.csv', index=False)


In [38]:
## THIS CODE IS TO FIND THE RMSE BETWEEN TRUE AND PREDICTED LST ##

true_df = pd.read_csv("test_set.csv")
pred_df = pd.read_csv("jw_pred_long_do.csv")

print(true_df.head())
print(pred_df.head())

merged_df = true_df.merge(pred_df, on=["x", "y", "Date"], suffixes=("_true", "_pred"))
print(merged_df.head())

# Compute RMSE
rmse = np.sqrt(np.mean((merged_df["Value"] - merged_df["Predicted_LST"]) ** 2))
print(f"RMSE between true and predicted LST for Jurong West is: {rmse:.4f}")

            x         y          Date      Value
0  103.964566  1.350459  Jan-Feb 2023  19.300148
1  103.964566  1.351269  Jan-Feb 2023  19.300060
2  103.964566  1.352079  Jan-Feb 2023  19.299971
3  103.964566  1.352889  Jan-Feb 2023  19.299883
4  103.965377  1.348838  Jan-Feb 2023  19.300281
            x         y          Date  Predicted_LST
0  103.964566  1.350459  Jan-Feb 2023      25.156872
1  103.964566  1.350459  Mar-Apr 2023      25.448523
2  103.964566  1.350459  May-Jun 2023      25.188898
3  103.964566  1.350459  Jul-Aug 2023      25.470753
4  103.964566  1.350459  Sep-Oct 2023      25.496620
            x         y          Date      Value  Predicted_LST
0  103.964566  1.350459  Jan-Feb 2023  19.300148      25.156872
1  103.964566  1.351269  Jan-Feb 2023  19.300060      25.159555
2  103.964566  1.352079  Jan-Feb 2023  19.299971      24.815573
3  103.964566  1.352889  Jan-Feb 2023  19.299883      25.040361
4  103.965377  1.348838  Jan-Feb 2023  19.300281      24.954197
RMSE